In [1]:
# FFSM++ Oracle paper output parser #

### Header ###

In [2]:
import sys          # only needed to determine Python version number
import pandas as pd
import numpy as np
from pandas import DataFrame, Series
import output_parser_lib as parserlib

# Matplotlib options
import matplotlib.pyplot as plt
import matplotlib   # only needed to determine Matplotlib version number
# Render plots inline:
%matplotlib inline  
pd.set_option('display.mpl_style', 'default') # Make the graphs a bit prettier
plt.rcParams['figure.figsize'] = (15, 5)

pd.set_option('display.max_rows', 88)


### Global variables ###

In [3]:
scenarios = ['constant', 'constant_nomgm', 'constant_nora', 'constant_carea', 'arpege_a1b', 'arpege_a1b_carea', 'arpege_a1b_noexp', 'arpege_a1b_fullexp', 'arpege_a1b_nora', 'arpege_a1b_nomgm', 'arpege_a1b_nopr', 'arpege_a1b_noph', 'arpege_a2', 'arpege_b1', 'CNCM33_1', 'DMIEH5C_1', 'DMIEH5C_2', 'DMIEH5C_3', 'EGMAM2_3', 'HADGEM2_1', 'IPCM4_1', 'IPCM4_2', 'IPCM4_3', 'MPEH5C_1', 'MPEH5C_2', 'MPEH5C_3']
scenarios_a1b = ['arpege_a1b', 'CNCM33_1', 'DMIEH5C_1', 'DMIEH5C_2', 'DMIEH5C_3', 'EGMAM2_3', 'HADGEM2_1', 'IPCM4_1', 'IPCM4_2', 'IPCM4_3', 'MPEH5C_1', 'MPEH5C_2', 'MPEH5C_3']
#scenarios = ['constant']

baseIDir = 'results'
regions_map = {11042: 'Alsace', 11061: 'Aquitaine', 11072: 'Auvergne', 11025: 'Basse-Normandie', 11026: 'Bourgogne',
    11052: 'Bretagne', 11024: 'Centre', 11021: 'Champagne-Ardenne', 11083: 'Corse', 11043: 'Franche-Comté',
    11023: 'Haute-Normandie', 11010: 'Île de France', 11081: 'Languedoc-Roussillon', 11063: 'Limousin',
    11041: 'Lorraine', 11062: 'Midi-Pyrénées', 11030: 'Nord - Pas-de-Calais', 11051: 'Pays de la Loire',
    11022: 'Picardie', 11053: 'Poitou-Charentes', 11082: 'Provence-Alpes-Côte d\'Azur', 11071: 'Rhône-Alpes'} 

regions_map_name_to_short = {'Alsace': 'AL', 'Aquitaine': 'AQ', 'Auvergne': 'AU', 'Basse-Normandie': 'BN',
                             'Bourgogne': 'BO', 'Bretagne': 'BR', 'Centre': 'CE', 'Champagne-Ardenne': 'CA',
                             'Corse': 'CO', 'Franche-Comté': 'FC', 'Haute-Normandie': 'HN', 'Île de France': 'IF',
                             'Languedoc-Roussillon': 'LR', 'Limousin': 'LI', 'Lorraine': 'LO', 'Midi-Pyrénées': 'MP',
                             'Nord - Pas-de-Calais': 'NP', 'Pays de la Loire': 'PL', 'Picardie': 'PI',
                             'Poitou-Charentes': 'PC', 'Provence-Alpes-Côte d\'Azur': 'PA', 'Rhône-Alpes': 'RA',
                             'France': 'Fr'}
regions_map_id_to_short = {11042: 'AL', 11061: 'AQ', 11072: 'AU', 11025: 'BN', 11026: 'BO', 11052: 'BR', 11024: 'CE',
                           11021: 'CA', 11083: 'CO', 11043: 'FC', 11023: 'HN', 11010: 'IF', 11081: 'LR', 11063: 'LI',
                           11041: 'LO', 11062: 'MP', 11030: 'NP', 11051: 'PL', 11022: 'PI', 11053: 'PC', 11082: 'PA',
                           11071: 'RA', 11000: 'France'}

products_map = {'hardWRoundW': 'Hardwood Roundwood', 'softWRoundW': 'Softwood Roundwood',
                'pulpWFuelW': 'Pulpwood and Fuelwood', 'fuelW': 'Fuelwood', 'hardWSawnW': 'Hardwood Sawnwood',
                'softWSawnW': 'Softwood Sawnwood', 'plyW': 'Plywood', 'pulpW': 'Pulpwood', 'pannels': 'Pannels'}

# sigma – prix elasticity (supply and demand)
sigma_map = {'hardWRoundW': 1.2, 'softWRoundW': 0.9, 'pulpWFuelW': 0.9, 'fuelW': -0.6, 'hardWSawnW': -0.5,
             'softWSawnW': -0.4, 'plyW': -0.3, 'pannels': -0.5, 'pulpW': -0.5}

fgroup_map = {'broadL_highF': 'Broadleaved','broadL_mixedF': 'Broadleaved','broadL_copp': 'Broadleaved',
             'con_highF': 'Coniferous', np.nan: 'All forest'}

gamma = 0.5
trs   = 0.5




### Definition of helper functions ##

In [4]:
def form_i(x):
    return "{:0,.0f}".format(x)  # e.g. 1234567.890 --> 1,234,567
def form_f2(x):
    return "{:0,.2f}".format(x)  # e.g. 1234567.890 --> 1,234,567.89
def form_f3(x):
    return "{:0,.3f}".format(x)  # e.g. 1234567.890 --> 1,234,567.890

def JoinMulti(DFList):
    FinalDF=DFList[0].reset_index()
    for OtherDF in DFList[1:]:
        FinalDF=pd.merge(FinalDF, OtherDF.reset_index(), how='outer')

    #Now I want reindex it so that it's indexed the same as the `DataFrame` with the highest number of levels
    NLevels=[x.index.nlevels for x in DFList]
    MaxIndexPos=NLevels.index(max(NLevels))
    FinalIndex=DFList[MaxIndexPos].index
    FinalDF=FinalDF.set_index(FinalIndex.names).reindex(FinalIndex)
    return FinalDF

### Data import ###
(may take long time)

In [5]:
forestData_byscenario = [pd.read_csv(baseIDir+'/forestData_'+scenario+'.csv',sep=';',  dtype={'value': np.float64}) for scenario in scenarios]
forestDataReg = pd.concat(forestData_byscenario)
forestDataReg = forestDataReg.drop(['Unnamed: 8'],axis=1) # we don't want this column never
forestDataReg = forestDataReg.drop(['country'],axis=1) # we don't want this column never
productData_byscenario = [pd.read_csv(baseIDir+'/productData_'+scenario+'.csv',sep=';',  dtype={'value': np.float64}) for scenario in scenarios]
productDataReg = pd.concat(productData_byscenario)
productDataReg = productDataReg.drop(['Unnamed: 8'],axis=1) # we don't want this column never
productDataReg = productDataReg.drop(['country'],axis=1) # we don't want this column never
carbonData_byscenario = [pd.read_csv(baseIDir+'/carbonBalance_'+scenario+'.csv',sep=';',  dtype={'value': np.float64}) for scenario in scenarios]
carbonDataReg = pd.concat(carbonData_byscenario)
carbonDataReg = carbonDataReg.drop(['Unnamed: 6'],axis=1) # we don't want this column never
carbonDataReg = carbonDataReg.drop(['country'],axis=1) # we don't want this column never

### Common data tidying ###

In [6]:
forestDataReg['spGroup'] = forestDataReg['forType']
forestDataReg.replace({'spGroup': fgroup_map}, inplace=True) # doesn't work on index, only on a column

#forestDataReg.replace({'region': regions_map}, inplace=True)
#productDataReg.replace({'region': regions_map}, inplace=True)
productDataReg['prodLev'] = np.where(productDataReg['prod'].isin(['hardWRoundW', 'softWRoundW', 'pulpWFuelW','ashRoundW']), 'priProd', 'secProd')

### Ecological - ecosystem services dimension ###

#### Area of total forest by region:  table now and 2100 (just to report regions with maximum variations) ####
Unit: ha

In [7]:
# creating tables forArea_total_arpege_a1b and forArea_total_a1b
forArea = forestDataReg[forestDataReg['parName'] == 'forArea'].copy()
#forArea.replace({'region': regions_map}, inplace=True) # doesn't work on index, only on a column
#forArea.replace({'region': regions_map_name_to_short}, inplace=True)
forArea.replace({'region': regions_map_id_to_short}, inplace=True)
forArea = forArea.drop(['freeDim','parName','forType'],axis=1) # we don't want these columns never
forArea_byGroup = forArea.groupby(['scen','region','year','spGroup'], as_index=False).agg(np.sum)
forArea_total = forArea.groupby(['scen','region','year'], as_index=False).agg(np.sum)
forArea_total = forArea_total[forArea_total['year'].isin([2015,2050,2100])]
forArea_total_arpege_a1b = forArea_total[forArea_total['scen'] == 'arpege_a1b']
forArea_total_arpege_a1b = forArea_total_arpege_a1b.drop(['scen'], axis=1)
forArea_total_arpege_a1b.set_index(['region','year'], inplace=True)
forArea_total_arpege_a1b = forArea_total_arpege_a1b.unstack(['year'])
forArea_total_arpege_a1b.columns = forArea_total_arpege_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL

forArea_total_arpege_a1b['diff'] = forArea_total_arpege_a1b[2100]-forArea_total_arpege_a1b[2015]
forArea_total_arpege_a1b.loc['France'] = forArea_total_arpege_a1b.sum()

forArea_total_a1b = forArea_total[forArea_total['scen'].isin(scenarios_a1b)]
forArea_total_a1b_2100 = forArea_total_a1b[forArea_total_a1b['year']==2100]
forArea_total_a1b_2100 = forArea_total_a1b_2100.groupby(['scen']).agg(np.sum)
#print(forArea_total_a1b_2100)

forArea_total_a1b = forArea_total_a1b.groupby(['region','year']).agg(np.mean)
forArea_total_a1b = forArea_total_a1b.unstack(['year'])
forArea_total_a1b.columns = forArea_total_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL

forArea_total_a1b['diff'] = forArea_total_a1b[2100]-forArea_total_a1b[2015]

forArea_total_a1b.loc['France'] = forArea_total_a1b.sum()
forArea_total_a1b_Mha = forArea_total_a1b/1000000
forArea_total_a1b_Mha['diff_pc'] = 100*forArea_total_a1b_Mha['diff']/forArea_total_a1b_Mha[2015]
print("Notes:")
print("These results are direct results of the work of Anne. We can ask her how much confident is she on these before deciding which exposition give to them (Appendix, main text..)")

forArea_total_a1b_Mha
#print (forArea_total_a1b.to_latex(formatters=[form1, form1, form1]))


Notes:
These results are direct results of the work of Anne. We can ask her how much confident is she on these before deciding which exposition give to them (Appendix, main text..)


year,2015,2050,2100,diff,diff_pc
region,,,,,
AL,0.313517,0.307432,0.298958,-0.014559,-4.643880
AQ,1.532109,1.515215,1.491478,-0.040632,-2.652001
AU,0.734735,0.727587,0.717509,-0.017226,-2.344579
BN,0.140962,0.135319,0.127682,-0.013280,-9.421127
BO,0.932854,0.912303,0.883781,-0.049073,-5.260569
BR,0.274563,0.265038,0.252028,-0.022535,-8.207594
CA,0.651349,0.619789,0.577461,-0.073889,-11.343923
CE,0.850686,0.830283,0.802033,-0.048653,-5.719313
CO,0.258457,0.265028,0.274703,0.016246,6.285734


#### Share of broadleaved/coniferous:  table now/2100 by region ####

In [8]:
forArea_byGroup_a1b = forArea_byGroup[forArea_byGroup['scen'].isin(scenarios_a1b)]
forArea_byGroup_a1b = forArea_byGroup_a1b[forArea_byGroup_a1b['year'].isin([2015,2100])]
forArea_byGroup_a1b = forArea_byGroup_a1b.groupby(['region','year','spGroup']).agg(np.mean)
#forArea_byGroup_a1b

In [9]:
#forArea_byGroup_arpege_a1b = forArea_byGroup[forArea_byGroup['scen'] == 'arpege_a1b']
#forArea_byGroup_arpege_a1b = forArea_byGroup_arpege_a1b.drop(['scen'], axis=1)
#forArea_byGroup_arpege_a1b = forArea_byGroup_arpege_a1b[forArea_byGroup_arpege_a1b['year'].isin([2015,2100])]
#forArea_byGroup_arpege_a1b.set_index(['region','year','spGroup'], inplace=True)
#forArea_byGroup_arpege_a1b=forArea_byGroup_arpege_a1b.unstack(['year','spGroup'])
#forArea_byGroup_arpege_a1b.columns = forArea_byGroup_arpege_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL
#forArea_byGroup_arpege_a1b.columns.set_levels([ [2015, 2100], ['br', 'con']], inplace=True)
#forArea_byGroup_arpege_a1b.loc['France'] = forArea_byGroup_arpege_a1b.sum()
#forArea_byGroup_arpege_a1b[2015,'Broadleaved'] = 100 * forArea_byGroup_arpege_a1b[2015,'br'] / (forArea_byGroup_arpege_a1b[2015,'br']+forArea_byGroup_arpege_a1b[2015,'con']) 
#forArea_byGroup_arpege_a1b[2015,'Coniferous'] = 100 * forArea_byGroup_arpege_a1b[2015,'con'] / (forArea_byGroup_arpege_a1b[2015,'br']+forArea_byGroup_arpege_a1b[2015,'con']) 
#forArea_byGroup_arpege_a1b[2100,'Broadleaved'] = 100 * forArea_byGroup_arpege_a1b[2100,'br'] / (forArea_byGroup_arpege_a1b[2100,'br']+forArea_byGroup_arpege_a1b[2100,'con']) 
#forArea_byGroup_arpege_a1b[2100,'Coniferous'] = 100 * forArea_byGroup_arpege_a1b[2100,'con'] / (forArea_byGroup_arpege_a1b[2100,'br']+forArea_byGroup_arpege_a1b[2100,'con']) 
#forArea_byGroup_arpege_a1b.drop([(2015,'br'),(2015,'con'),(2100,'br'),(2100,'con')], axis=1, inplace=True)
#forArea_byGroup_arpege_a1b
#print (forArea_byGroup_arpege_a1b.to_latex(formatters=[form2, form2, form2, form2]))

forArea_byGroup_a1b = forArea_byGroup[forArea_byGroup['scen'].isin(scenarios_a1b)]
forArea_byGroup_a1b = forArea_byGroup_a1b[forArea_byGroup_a1b['year'].isin([2015,2100])]
forArea_byGroup_a1b = forArea_byGroup_a1b.groupby(['region','year','spGroup']).agg(np.mean)
forArea_byGroup_a1b = forArea_byGroup_a1b.unstack(['year','spGroup'])
forArea_byGroup_a1b.columns = forArea_byGroup_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL
forArea_byGroup_a1b.columns.set_levels([ [2015, 2100], ['br', 'con']], inplace=True)
forArea_byGroup_a1b.loc['France'] = forArea_byGroup_a1b.sum()
forAreaShare_byGroup_a1b = forArea_byGroup_a1b.copy()
forAreaShare_byGroup_a1b[2015,'Broadleaved'] = 100 * forArea_byGroup_a1b[2015,'br']  / (forArea_byGroup_a1b[2015,'br']+forArea_byGroup_a1b[2015,'con']) 
forAreaShare_byGroup_a1b[2015,'Coniferous']  = 100 * forArea_byGroup_a1b[2015,'con'] / (forArea_byGroup_a1b[2015,'br']+forArea_byGroup_a1b[2015,'con']) 
forAreaShare_byGroup_a1b[2100,'Broadleaved'] = 100 * forArea_byGroup_a1b[2100,'br']  / (forArea_byGroup_a1b[2100,'br']+forArea_byGroup_a1b[2100,'con']) 
forAreaShare_byGroup_a1b[2100,'Coniferous']  = 100 * forArea_byGroup_a1b[2100,'con'] / (forArea_byGroup_a1b[2100,'br']+forArea_byGroup_a1b[2100,'con']) 
forAreaShare_byGroup_a1b.drop([(2015,'br'),(2015,'con'),(2100,'br'),(2100,'con')], axis=1, inplace=True)
#print (forArea_byGroup_a1b.to_latex(formatters=[form2, form2, form2, form2]))
print("Notes:")
print("- The constant share is really a coincidence and depends from the two opposite tendencies of cc of favour broadleaved area and management of favour con area.")
#print(forAreaShare_byGroup_a1b)
forArea_byGroup_a1b[2015,'Total'] = forArea_byGroup_a1b[2015,'br'] + forArea_byGroup_a1b[2015,'con']
forArea_byGroup_a1b[2100,'Total'] = forArea_byGroup_a1b[2100,'br'] + forArea_byGroup_a1b[2100,'con']
forArea_byGroup_a1b = forArea_byGroup_a1b / 1000000 # ha -> Mha
forArea_byGroup_a1b.sortlevel('year', axis=1, inplace=True)
forArea_byGroup_a1b.columns.set_levels([ [2015, 2100], ['Br', 'Con', 'Total']], inplace=True)
#print (forArea_byGroup_a1b.to_latex(formatters=[form_f3, form_f3, form_f3, form_f3, form_f3, form_f3]))
forArea_byGroup_a1b


Notes:
- The constant share is really a coincidence and depends from the two opposite tendencies of cc of favour broadleaved area and management of favour con area.


year         2015                           2100                     
spGroup        Br       Con      Total        Br       Con      Total
region                                                               
AL       0.194378  0.119139   0.313517  0.184779  0.114179   0.298958
AQ       0.733026  0.799084   1.532109  0.878139  0.613339   1.491478
AU       0.387465  0.347270   0.734735  0.412846  0.304663   0.717509
BN       0.115459  0.025503   0.140962  0.098431  0.029250   0.127682
BO       0.831256  0.101598   0.932854  0.781467  0.102314   0.883781
BR       0.190646  0.083917   0.274563  0.178041  0.073987   0.252028
CA       0.593213  0.058136   0.651349  0.485865  0.091596   0.577461
CE       0.735750  0.114936   0.850686  0.660184  0.141849   0.802033
CO       0.206745  0.051712   0.258457  0.224792  0.049911   0.274703
FC       0.528690  0.175674   0.704364  0.512569  0.160835   0.673404
HN       0.195093  0.020679   0.215772  0.174825  0.019043   0.193868
IF       0.249195  0.020326   0.269521  0.196659  0.047292   0.243951
LI       0.382653  0.177657   0.560310  0.372619  0.172867   0.545486
LO       0.608484  0.203547   0.812031  0.552621  0.225491   0.778112
LR       0.577455  0.327166   0.904621  0.594089  0.317613   0.911701
MP       1.041194  0.149062   1.190256  1.003341  0.185945   1.189286
NP       0.082749  0.002600   0.085349  0.074544  0.001942   0.076487
PA       0.451154  0.633559   1.084713  0.564970  0.570687   1.135657
PC       0.301375  0.051734   0.353109  0.256824  0.082206   0.339030
PI       0.298404  0.013134   0.311538  0.251926  0.025603   0.277529
PL       0.180971  0.084081   0.265052  0.151271  0.088732   0.240003
RA       0.876197  0.739980   1.616178  0.930004  0.734248   1.664252
France   9.761551  4.300498  14.062049  9.540807  4.153591  13.694398

#### Vol/ha of forest: map now/2100 by pixel, table now/2100 by region ####
vol unit: Mm^3, area unit: ha, v/ha unit: m^3/ha    

In [10]:
# (1/2) 
forVol = forestDataReg[forestDataReg['parName'] == 'vol'].copy()
forVol.replace({'region': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column
forVol.drop(['freeDim','parName','forType'],axis=1, inplace=True) # we don't want these columns never
forVol_byGroup = forVol.groupby(['scen','region','year','spGroup'], as_index=False).agg(np.sum)
forVol_total = forVol.groupby(['scen','region','year'], as_index=False).agg(np.sum)
forVol_total = forVol_total[forVol_total['year'].isin([2015,2100])]

forVol_total_arpege_a1b = forVol_total[forVol_total['scen'] == 'arpege_a1b']
forVol_total_arpege_a1b = forVol_total_arpege_a1b.drop(['scen'], axis=1)
forVol_total_arpege_a1b.set_index(['region','year'], inplace=True)
forVol_total_arpege_a1b = forVol_total_arpege_a1b.unstack(['year'])
forVol_total_arpege_a1b.columns = forVol_total_arpege_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL
forVol_total_arpege_a1b['diff'] = forVol_total_arpege_a1b[2100]-forVol_total_arpege_a1b[2015]
forVol_total_arpege_a1b.loc['France'] = forVol_total_arpege_a1b.sum()
forVol_total_arpege_a1b

forVol_total_a1b = forVol_total[forVol_total['scen'].isin(scenarios_a1b)]
forVol_total_a1b = forVol_total_a1b.groupby(['region','year']).agg(np.mean)
forVol_total_a1b = forVol_total_a1b.unstack(['year'])
forVol_total_a1b.columns = forVol_total_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL
forVol_total_a1b['diff'] = forVol_total_a1b[2100]-forVol_total_a1b[2015]
forVol_total_a1b.loc['France'] = forVol_total_a1b.sum()
forVol_total_a1b['diff_pc'] = 100*forVol_total_a1b['diff']/forVol_total_a1b[2015]
forVol_total_a1b

year,2015,2100,diff,diff_pc
region,,,,
AL,90.141152,210.881660,120.740508,133.946046
AQ,334.227454,615.041154,280.813700,84.018741
AU,205.431432,419.721031,214.289598,104.311982
BN,35.107366,31.421692,-3.685674,-10.498292
BO,222.412099,461.211808,238.799708,107.368128
BR,82.911446,172.723269,89.811823,108.322587
CA,147.891249,173.940262,26.049013,17.613627
CE,191.314400,425.786692,234.472292,122.558622
CO,38.025088,78.792969,40.767881,107.213112


In [11]:
# (2/2)
tempVol = forVol_total_a1b.copy()
tempVol.drop('diff_pc', axis=1, inplace=True)
columnIdx = pd.MultiIndex.from_tuples([('vol',2015),('vol',2100),('vol','diff')], names=['var', 'year'])                       
tempVol.columns=columnIdx  
tempArea =forArea_total_a1b.copy()
columnIdx = pd.MultiIndex.from_tuples([('area',2015),('area',2050),('area',2100),('area','diff')], names=['var', 'year'])                       
tempArea.columns=columnIdx  
forVolOverArea_total_a1b = pd.concat([tempVol,tempArea], axis=1)
forVolOverArea_total_a1b['v/ha',2015]= 1000000*forVolOverArea_total_a1b['vol',2015] / forVolOverArea_total_a1b['area',2015] 
forVolOverArea_total_a1b['v/ha',2100]= 1000000*forVolOverArea_total_a1b['vol',2100] / forVolOverArea_total_a1b['area',2100] 
forVolOverArea_total_a1b['diff_pc',2100] = 100*(forVolOverArea_total_a1b['v/ha',2100]-forVolOverArea_total_a1b['v/ha',2015])/forVolOverArea_total_a1b['v/ha',2015]
forVolOverArea_total_a1b.drop([('vol',2015),('vol',2100),('vol','diff'),('area',2015),('area',2050),('area',2100),('area','diff')], axis=1, inplace=True)
forVolOverArea_total_a1b
#print (forVolOverArea_total_arpege_a1b.to_latex(formatters=[form2, form2]))

var           v/ha                 diff_pc
year          2015        2100        2100
region                                    
AL      287.515700  705.389030  145.339308
AQ      218.148544  412.370270   89.031869
AU      279.599287  584.969928  109.217246
BN      249.056011  246.094234   -1.189201
BO      238.421063  521.862217  118.882598
BR      301.975849  685.333274  126.949697
CA      227.053622  301.215676   32.662793
CE      224.894292  530.884587  136.059609
CO      147.123286  286.829407   94.958537
FC      259.227809  567.126746  118.775427
HN      235.918829  307.875368   30.500549
IF      189.091970  230.292690   21.788720
LI      244.411267  310.440815   27.015755
LO      272.530325  506.561428   85.873417
LR      132.232143  169.747523   28.370847
MP      197.041986  573.578949  191.094786
NP      302.113878  717.419106  137.466452
PA      143.384519  190.464963   32.835096
PC      198.310220  393.901557   98.628975
PI      187.048491   73.966061  -60.456211
PL      267.155078  499.671126   87.034111
RA      233.055394  588.710847  152.605544
France  220.551375  439.058241   99.073001

#### Forest volumes share by forest type now and 2100 ####

In [12]:
forVol_byGroup
forVol_byGroup_a1b = forVol_byGroup[forVol_byGroup['scen'].isin(scenarios_a1b)]
forVol_byGroup_a1b = forVol_byGroup_a1b[forVol_byGroup_a1b['year'].isin([2015,2100])]
forVol_byGroup_a1b = forVol_byGroup_a1b.groupby(['region','year','spGroup']).agg(np.mean)
forVol_byGroup_a1b = forVol_byGroup_a1b.unstack(['year','spGroup'])
forVol_byGroup_a1b.columns = forVol_byGroup_a1b.columns.droplevel() # DROP TOP COLUMN LEVEL
forVol_byGroup_a1b.columns.set_levels([ [2015, 2100], ['br', 'con']], inplace=True)
forVol_byGroup_a1b.loc['France'] = forVol_byGroup_a1b.sum()
forVol_byGroup_a1b
forVolShare_byGroup_a1b = forVol_byGroup_a1b.copy()
forVolShare_byGroup_a1b[2015,'Broadleaved'] = 100 * forVol_byGroup_a1b[2015,'br']  / (forVol_byGroup_a1b[2015,'br']+forVol_byGroup_a1b[2015,'con']) 
forVolShare_byGroup_a1b[2015,'Coniferous']  = 100 * forVol_byGroup_a1b[2015,'con'] / (forVol_byGroup_a1b[2015,'br']+forVol_byGroup_a1b[2015,'con']) 
forVolShare_byGroup_a1b[2100,'Broadleaved'] = 100 * forVol_byGroup_a1b[2100,'br']  / (forVol_byGroup_a1b[2100,'br']+forVol_byGroup_a1b[2100,'con']) 
forVolShare_byGroup_a1b[2100,'Coniferous']  = 100 * forVol_byGroup_a1b[2100,'con'] / (forVol_byGroup_a1b[2100,'br']+forVol_byGroup_a1b[2100,'con']) 
forVolShare_byGroup_a1b.drop([(2015,'br'),(2015,'con'),(2100,'br'),(2100,'con')], axis=1, inplace=True)
#forVolShare_byGroup_a1b
forVol_byGroup_a1b[2015,'Total'] = forVol_byGroup_a1b[2015,'br'] + forVol_byGroup_a1b[2015,'con']
forVol_byGroup_a1b[2100,'Total'] = forVol_byGroup_a1b[2100,'br'] + forVol_byGroup_a1b[2100,'con']
#forVol_byGroup_a1b = forVol_byGroup_a1b / 1000000 
forVol_byGroup_a1b.sortlevel('year', axis=1, inplace=True)
forVol_byGroup_a1b.columns.set_levels([ [2015, 2100], ['Br', 'Con', 'Total']], inplace=True)
print (forVol_byGroup_a1b.to_latex(formatters=[form_f2, form_f2, form_f2, form_f2, form_f2, form_f2]))
forVol_byGroup_a1b

\begin{tabular}{lrrrrrr}
\toprule
year &     2015 &          &          &     2100 &          &          \\
spGroup &       Br &      Con &    Total &       Br &      Con &    Total \\
region &          &          &          &          &          &          \\
\midrule
AL     &    50.80 &    39.34 &    90.14 &   126.82 &    84.06 &   210.88 \\
AQ     &   158.51 &   175.72 &   334.23 &   445.12 &   169.92 &   615.04 \\
AU     &    99.24 &   106.19 &   205.43 &   278.13 &   141.59 &   419.72 \\
BN     &    27.43 &     7.68 &    35.11 &    28.26 &     3.16 &    31.42 \\
BO     &   181.80 &    40.61 &   222.41 &   437.63 &    23.59 &   461.21 \\
BR     &    56.33 &    26.58 &    82.91 &   150.06 &    22.66 &   172.72 \\
CA     &   121.73 &    26.16 &   147.89 &   151.57 &    22.37 &   173.94 \\
CE     &   154.86 &    36.45 &   191.31 &   363.04 &    62.75 &   425.79 \\
CO     &    27.10 &    10.92 &    38.03 &    50.40 &    28.40 &    78.79 \\
FC     &   121.49 &    61.10 &   182.59 &   31

year            2015                                   2100               \
spGroup           Br          Con        Total           Br          Con   
region                                                                     
AL         50.798359    39.342792    90.141152   126.817606    84.064054   
AQ        158.510685   175.716769   334.227454   445.124077   169.917077   
AU         99.237740   106.193692   205.431432   278.129954   141.591077   
BN         27.432360     7.675006    35.107366    28.257484     3.164208   
BO        181.802199    40.609900   222.412099   437.625308    23.586500   
BR         56.329538    26.581908    82.911446   150.062492    22.660777   
CA        121.731911    26.159338   147.891249   151.571270    22.368992   
CE        154.862408    36.451992   191.314400   363.040785    62.745908   
CO         27.100827    10.924262    38.025088    50.396538    28.396431   
FC        121.490342    61.100338   182.590680   313.575115    68.330546   
HN         43.781102     7.123581    50.904683    57.655454     2.031729   
IF         46.803920     4.160415    50.964335    46.288677     9.891510   
LI         90.757271    46.188900   136.946171   135.725085    33.616008   
LO        144.917180    76.386023   221.303203   286.109117   108.052638   
LR         61.103858    58.516115   119.619974   107.528408    47.230631   
MP        187.006931    47.523392   234.530323   561.011400   121.137846   
NP         24.191038     1.594228    25.785265    54.553732     0.319283   
PA         58.739015    96.791992   155.531008   134.649262    81.653592   
PC         58.189369    11.835815    70.025185   107.772838    25.771631   
PI         52.347177     5.925558    58.272735    18.699146     1.828583   
PL         47.301435    23.508654    70.810088    90.742246    29.180515   
RA        184.000592   192.658308   376.658900   629.161769   350.601154   
France   1998.435257  1102.968981  3101.404238  4574.497762  1438.140691   

year                  
spGroup        Total  
region                
AL        210.881660  
AQ        615.041154  
AU        419.721031  
BN         31.421692  
BO        461.211808  
BR        172.723269  
CA        173.940262  
CE        425.786692  
CO         78.792969  
FC        381.905662  
HN         59.687183  
IF         56.180187  
LI        169.341092  
LO        394.161755  
LR        154.759038  
MP        682.149246  
NP         54.873014  
PA        216.302854  
PC        133.544469  
PI         20.527729  
PL        119.922762  
RA        979.762923  
France   6012.638453

## Forest owners dimension ##
#### Forest expected returns: by region 2015 constant vs a1b and (next table) national ones a1b time trend. ####

In [48]:

# ATTENCTION: This table has a lot of useless code, because initially I tried to keep the computation of "All forest"
# expected returns with it original "special" meaning (no ra and best ft on each plot), while now it is a simply
# weighted average of the expectedReturns of the various ftypes (by area).


#forestDataReg
#scen 	parName 	region 	forType 	freeDim 	year 	value 	spGroup
#expReturns = forestDataReg[(forestDataReg['parName'] == 'expReturns') & (forestDataReg['forType'].isnull())].copy() # don't like and
expReturns = forestDataReg[(forestDataReg['parName'] == 'expReturns') | (forestDataReg['parName'] == 'forArea')].copy() 
del expReturns['freeDim']
expReturns.replace({'region': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column
expReturns_2015 = expReturns[expReturns['year'].isin([2015])]
expReturns_2015 = expReturns_2015.fillna('allForest')
expReturns_2015_a1b = expReturns_2015[expReturns_2015['scen'].isin(scenarios_a1b)]
expReturns_2015_a1b = expReturns_2015_a1b.groupby(['region','spGroup','forType','parName']).agg(np.mean) # average by a1b scenarios
expReturns_2015_a1b['scenario'] = 'a1b'
expReturns_2015_a1b.set_index('scenario', append=True, inplace=True)

expReturns_2015_constant = expReturns_2015[expReturns_2015['scen'].isin(['constant'])]
expReturns_2015_constant = expReturns_2015_constant.groupby(['region','spGroup','forType','parName']).agg(np.mean) # don't do nothing, just setting the format the same as for a1b scenario
expReturns_2015_constant['scenario'] = 'constant'
expReturns_2015_constant.set_index('scenario', append=True, inplace=True)

expReturns_2015_a1b = expReturns_2015_a1b.append(expReturns_2015_constant)
expReturns_2015_a1b = expReturns_2015_a1b.drop(['year'],axis=1) # we don't want these columns never

expReturns_2015_a1b = expReturns_2015_a1b.unstack(['parName'])  
expReturns_2015_a1b.columns = expReturns_2015_a1b.columns.droplevel()

totfarea = expReturns_2015_a1b.copy()
del totfarea['expReturns']
totfarea =totfarea.fillna(0)
totfarea = totfarea.reset_index()
totfarea = totfarea.groupby(['region','scenario']).agg(np.sum) # grouping by forType

totfarea['spGroup'] = 'All forest'
totfarea.set_index('spGroup', append=True, inplace=True)
totfarea.rename(columns = {'forArea':'forArea2'}, inplace = True)

expReturns_2015_a1b = JoinMulti([expReturns_2015_a1b,totfarea])
expReturns_2015_a1b.rename(columns = {'forArea':'forArea1'}, inplace = True)
expReturns_2015_a1b = expReturns_2015_a1b.fillna(0)
expReturns_2015_a1b['forArea'] = expReturns_2015_a1b['forArea1']+expReturns_2015_a1b['forArea2']
expReturns_2015_a1b['sumExpReturns'] =  expReturns_2015_a1b['expReturns'] * expReturns_2015_a1b['forArea'] # verified equal to the sumExpReturns computed in C++
expReturns_2015_a1b = expReturns_2015_a1b.drop(['expReturns','forArea1','forArea2'],axis=1) 
expReturns_2015_a1b = expReturns_2015_a1b.reset_index()

expReturns_2015_a1b = expReturns_2015_a1b.groupby(['region','spGroup','scenario']).agg(sum) # sum by specific forType
expReturns_2015_a1b['expReturns'] = expReturns_2015_a1b['sumExpReturns'] / expReturns_2015_a1b['forArea'] 
#expReturns_2015_a1b = expReturns_2015_a1b.drop(['sumExpReturns','forArea'],axis=1) 
expReturns_2015_a1b = expReturns_2015_a1b.unstack('spGroup')
expReturns_2015_a1b = expReturns_2015_a1b.unstack('scenario')
expReturns_2015_a1b.loc['France'] = expReturns_2015_a1b.sum()
expReturns_2015_a1b.loc['France',('expReturns','All forest','a1b')] = expReturns_2015_a1b.loc['France',('sumExpReturns','All forest','a1b')] / expReturns_2015_a1b.loc['France',('forArea','All forest','a1b')]
expReturns_2015_a1b.loc['France',('expReturns','All forest','constant')] = expReturns_2015_a1b.loc['France',('sumExpReturns','All forest','constant')] / expReturns_2015_a1b.loc['France',('forArea','All forest','constant')]
expReturns_2015_a1b.loc['France',('expReturns','Broadleaved','a1b')] = expReturns_2015_a1b.loc['France',('sumExpReturns','Broadleaved','a1b')] / expReturns_2015_a1b.loc['France',('forArea','Broadleaved','a1b')]
expReturns_2015_a1b.loc['France',('expReturns','Broadleaved','constant')] = expReturns_2015_a1b.loc['France',('sumExpReturns','Broadleaved','constant')] / expReturns_2015_a1b.loc['France',('forArea','Broadleaved','constant')]
expReturns_2015_a1b.loc['France',('expReturns','Coniferous','a1b')] = expReturns_2015_a1b.loc['France',('sumExpReturns','Coniferous','a1b')] / expReturns_2015_a1b.loc['France',('forArea','Coniferous','a1b')]
expReturns_2015_a1b.loc['France',('expReturns','Coniferous','constant')] = expReturns_2015_a1b.loc['France',('sumExpReturns','Coniferous','constant')] / expReturns_2015_a1b.loc['France',('forArea','Coniferous','constant')]

# Hoverriding the computation of the expReturns, no longer as from the C++ "forest" speacial meaning expected returns
# (placing the best ft in each pixel and not considering ra), but simply as a weighted average of the forest types (by area) 
expReturns_2015_a1b[('expReturns','All forest','a1b')] = (expReturns_2015_a1b[('expReturns','Broadleaved','a1b')]*expReturns_2015_a1b[('forArea','Broadleaved','a1b')]+expReturns_2015_a1b[('expReturns','Coniferous','a1b')]*expReturns_2015_a1b[('forArea','Coniferous','a1b')]) / (expReturns_2015_a1b[('forArea','Broadleaved','a1b')] + expReturns_2015_a1b[('forArea','Coniferous','a1b')])
expReturns_2015_a1b[('expReturns','All forest','constant')] = (expReturns_2015_a1b[('expReturns','Broadleaved','constant')]*expReturns_2015_a1b[('forArea','Broadleaved','constant')]+expReturns_2015_a1b[('expReturns','Coniferous','constant')]*expReturns_2015_a1b[('forArea','Coniferous','constant')]) / (expReturns_2015_a1b[('forArea','Broadleaved','constant')] + expReturns_2015_a1b[('forArea','Coniferous','constant')])

expReturns_2015_a1b.drop([('forArea','All forest', 'a1b'),
                          ('forArea','All forest', 'constant'),
                          ('forArea','Broadleaved', 'a1b'),
                          ('forArea','Broadleaved', 'constant'),
                          ('forArea','Coniferous', 'a1b'),
                          ('forArea','Coniferous', 'constant'),
                          ('sumExpReturns','All forest', 'a1b'),
                          ('sumExpReturns','All forest', 'constant'),
                          ('sumExpReturns','Broadleaved', 'a1b'),
                          ('sumExpReturns','Broadleaved', 'constant'),
                          ('sumExpReturns','Coniferous', 'a1b'),
                          ('sumExpReturns','Coniferous', 'constant')],axis=1, inplace=True)



expReturns_2015_a1b.columns = expReturns_2015_a1b.columns.droplevel()

print (expReturns_2015_a1b.to_latex(formatters=[form_f2, form_f2, form_f2, form_f2, form_f2, form_f2]))
expReturns_2015_a1b


\begin{tabular}{lrrrrrr}
\toprule
spGroup & All forest &          & Broadleaved &          & Coniferous &          \\
scenario &        a1b & constant &         a1b & constant &        a1b & constant \\
region &            &          &             &          &            &          \\
\midrule
AL     &     121.62 &    51.48 &       76.56 &    36.32 &     195.14 &    76.10 \\
AQ     &      84.32 &    45.70 &       39.55 &    19.96 &     125.40 &    69.21 \\
AU     &      85.56 &    43.03 &       40.64 &    17.46 &     135.69 &    71.37 \\
BN     &     135.19 &    72.62 &      103.48 &    52.79 &     278.74 &   160.10 \\
BO     &      90.24 &    41.92 &       74.40 &    29.56 &     219.85 &   139.95 \\
BR     &     103.22 &    51.09 &       72.62 &    36.71 &     172.75 &    83.28 \\
CA     &     150.32 &    63.28 &      132.75 &    53.14 &     329.62 &   159.72 \\
CE     &      43.78 &    22.33 &       27.15 &    13.74 &     150.21 &    75.72 \\
CO     &      81.43 &    18.33 &       82

spGroup   All forest             Broadleaved              Coniferous  \
scenario         a1b    constant         a1b    constant         a1b   
region                                                                 
AL        121.623817   51.475745   76.562480   36.324413  195.142154   
AQ         84.322534   45.704800   39.545565   19.959833  125.397923   
AU         85.560127   43.026712   40.635010   17.458366  135.685154   
BN        135.185375   72.619368  103.475993   52.792912  278.740538   
BO         90.240305   41.923870   74.398853   29.563477  219.851615   
BR        103.221530   51.086204   72.617836   36.710288  172.748077   
CA        150.322892   63.275682  132.751406   53.135291  329.620308   
CE         43.780483   22.334945   27.154702   13.742674  150.208308   
CO         81.434429   18.331545   82.881614    9.346982   75.648577   
FC        118.960710   47.126897  118.683975   49.396163  119.793538   
HN        138.340751   74.785442  127.750235   67.239059  238.252769   
IF         23.505402   13.886308   12.850512    8.175543  154.132900   
LI         66.899511   33.835680   32.127515   15.066477  141.794231   
LO        113.831140   48.499136  104.049783   45.892397  143.071538   
LR         14.311406    5.645047    7.880782    2.920863   25.661577   
MP         52.522626   21.665469   34.557317   14.606559  178.009769   
NP        195.290458   97.714560  199.867961   99.337026   49.631262   
PA         27.006616    8.546859   35.173802    9.732125   21.190808   
PC         33.675204   17.336067   11.638870    6.270329  162.046077   
PI        232.183127  123.109003  231.061868  120.093269  257.657846   
PL        168.548550   82.472927   91.608183   45.148046  334.150308   
RA         49.126822   20.930921   24.642259   11.513815   78.118554   
France     78.052967   36.082390   62.813828   27.745028  112.643753   

spGroup               
scenario    constant  
region                
AL         76.101800  
AQ         69.208200  
AU         71.374900  
BN        160.096000  
BO        139.951000  
BR         83.277100  
CA        159.721000  
CE         75.723400  
CO         54.157600  
FC         40.329900  
HN        143.164000  
IF         78.257000  
LI         74.029100  
LO         56.247600  
LR         10.442500  
MP         71.169500  
NP         49.635500  
PA          7.716400  
PC         80.910800  
PI        186.214000  
PL        161.714000  
RA         32.072400  
France     54.887943

In [49]:
# expReturns [a1b, national] time trend

expReturns_trend = forestDataReg[(forestDataReg['parName'] == 'forArea') | (forestDataReg['parName'] == 'sumExpReturns')].copy() 
del expReturns_trend['freeDim']
expReturns_trend.replace({'region': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column
expReturns_trend = expReturns_trend[expReturns_trend['year'].isin([2015,2040,2060,2080,2100])]
expReturns_trend = expReturns_trend[expReturns_trend['scen'].isin(['constant'])]
expReturns_trend = expReturns_trend.groupby(['region','spGroup','forType','parName','year']).agg(np.mean) # average by a1b scenarios
expReturns_trend = expReturns_trend.reset_index()
expReturns_trend = expReturns_trend.groupby(['spGroup','parName','year']).agg(np.sum) # sum by region and specific fType

expReturns_trend = expReturns_trend.unstack('parName')
expReturns_trend = expReturns_trend.unstack('year')
expReturns_trend.loc['All Forest'] = expReturns_trend.sum()
expReturns_trend = expReturns_trend.stack('year')
expReturns_trend.columns = expReturns_trend.columns.droplevel()
expReturns_trend['expReturns'] = expReturns_trend['sumExpReturns']/expReturns_trend['forArea']

expReturns_trend.drop(['forArea','sumExpReturns'],axis=1, inplace=True)
expReturns_trend = expReturns_trend.unstack('year')
expReturns_trend.columns = expReturns_trend.columns.droplevel()

#expReturns_tot_a1b['diff_pc',2100] = 100*(expReturns_tot_a1b['expReturns',2100]-expReturns_tot_a1b['expReturns',2015])/expReturns_tot_a1b['expReturns',2015]
print("Notes:")
print("- The expected returns of new forest investments growth in particular for the higher expected timber prices. It doesn't growth much from now to 2100 as due to the long forest rotation expected prices of timber for today plantation already incorporate future price increases.")
print("- Comparation with expected returns in 2015 without considering cc is available in the other PDF, Tab 2 \"Future forest relavie to current forest [2100 vs 2015] (2015 is constant)\"")
print (expReturns_trend.to_latex(formatters=[form_f2, form_f2, form_f2, form_f2, form_f2]))
expReturns_trend

Notes:
- The expected returns of new forest investments growth in particular for the higher expected timber prices. It doesn't growth much from now to 2100 as due to the long forest rotation expected prices of timber for today plantation already incorporate future price increases.
- Comparation with expected returns in 2015 without considering cc is available in the other PDF, Tab 2 "Future forest relavie to current forest [2100 vs 2015] (2015 is constant)"
\begin{tabular}{lrrrrr}
\toprule
year &  2015 &  2040 &  2060 &  2080 &  2100 \\
spGroup     &       &       &       &       &       \\
\midrule
Broadleaved & 27.75 & 23.25 & 22.02 & 21.81 & 22.13 \\
Coniferous  & 54.89 & 47.72 & 47.56 & 49.16 & 51.27 \\
All Forest  & 36.08 & 30.83 & 30.06 & 30.61 & 31.75 \\
\bottomrule
\end{tabular}



year,2015,2040,2060,2080,2100
spGroup,,,,,
Broadleaved,27.745031,23.245535,22.017850,21.808211,22.133010
Coniferous,54.887938,47.716273,47.564593,49.163010,51.265305
All Forest,36.082391,30.830843,30.057212,30.608931,31.754214


### HWP markets dimension ###

#### Market balance (production, consumption, import and export) and weighted price by primary and secondary product: table now and 2100 by region ###


In [15]:
rtData      = productDataReg[productDataReg['parName'] == 'rt'].copy() # alone as it takes some times...
rtData.drop('parName', axis=1, inplace=True)

In [16]:
rtData_a1b = rtData[rtData['scen'].isin(scenarios_a1b)]
rtData_a1b = rtData_a1b[rtData_a1b['year'].isin([2015,2100])]
rtData_a1b = rtData_a1b.groupby(['region','prod','freeDim','prodLev','year']).agg(np.mean)
rtData_a1b.reset_index(inplace=True)
rtData_a1b.replace({'region': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column
rtData_a1b.replace({'freeDim': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column
rtData_a1b.drop('prod', axis=1, inplace=True)
rtData_a1b_out = rtData_a1b.groupby(['region','prodLev','year']).agg(np.sum)
rtData_a1b_out = rtData_a1b_out.unstack(['year'])
rtData_a1b_out.columns=pd.MultiIndex.from_tuples([('rt_out',2015),('rt_out',2100)], names=['var', 'year'])  
rtData_a1b_in = rtData_a1b.groupby(['freeDim','prodLev','year']).agg(np.sum)
rtData_a1b_in = rtData_a1b_in.unstack(['year'])
rtData_a1b_in.columns=pd.MultiIndex.from_tuples([('rt_in',2015),('rt_in',2100)], names=['var', 'year'])  
#rtData_a1b_in

#### Total surplus, producer surplus and consumer surplus: table by region now and 2100 ####

In [17]:
balanceData_temp = productDataReg.drop(['freeDim'],axis=1) # we don't want this column never
balanceData_temp = balanceData_temp[balanceData_temp['parName'] != 'rt'] # we don't want rt in national data
#balanceData_temp = balanceData_temp[balanceData_temp['year'].isin([2014,2015,2099,2100])]
balanceData_temp.replace({'region': regions_map_id_to_short}, inplace=True) # doesn't work on index, only on a column

In [18]:
balanceData = balanceData_temp.copy()
# keeping all scenarios as we'll later need the surplus for different scenarios
#balanceData_a1b = balanceData[balanceData['scen'].isin(scenarios_a1b)]
#balanceData_a1b = balanceData_a1b[balanceData_a1b['year'].isin([2015,2100])]
#balanceData_a1b = balanceData_a1b.groupby(['region','prod','prodLev','year','parName']).agg(np.mean)
#balanceData.reset_index(inplace=True)

balanceData.set_index(['scen','region','prod','prodLev','year','parName'], inplace=True)
balanceData=balanceData.unstack(['parName'])
prods  = balanceData.index.get_level_values('prod').values

balanceData.columns = balanceData.columns.droplevel() # DROP TOP COLUMN LEVEL

balanceData['sigma']   = np.vectorize(sigma_map.get)(prods) # vectorize the get method of the dictionary and pass the prods ndarray
balanceData['rev'] = balanceData['st']*balanceData['pw']  
balanceData['exp'] = balanceData['dt']*balanceData['pl']  
#balanceData['rev2'] = balanceData['sl']*balanceData['pl']+balanceData['sa']*balanceData['pw'] # Attenction, for some reasons this is slighly different than rev

lagged_balanceData = balanceData.groupby(level=['scen','region','prod','prodLev']).shift()

sigma =        balanceData['sigma']
pc    =        balanceData['pc']
pc_l  = lagged_balanceData['pc']
sc    =        balanceData['sc']*1000000
sc_l  = lagged_balanceData['sc']*1000000
dc    =        balanceData['dc']*1000000
dc_l  = lagged_balanceData['dc']*1000000
inv   =        balanceData['in']*1000000
inv_l = lagged_balanceData['in']*1000000

balanceData['surpl_pr']  =  (sc * pc) - \
                            (pc_l /  pow(sc_l,1/sigma)) \
                           * pow(inv_l/inv,gamma/sigma) \
                           * pow(sc,(1+sigma)/sigma) \
                           * (sigma / (1+sigma))

balanceData['surpl_co']  =    (pc_l*pow(dc_l,-1/sigma)*sigma/(sigma+1)) \
                            * (pow(dc,(sigma+1)/sigma)-pow(trs*dc,(sigma+1)/sigma)) \
                            - (1-trs) * dc * pc   
balanceData['surpl_tot'] =  balanceData.fillna(0)['surpl_pr']+ balanceData.fillna(0)['surpl_co']        


#balanceData = balanceData[balanceData.index.get_level_values('year').isin([2015,2100])]

#balanceData

In [19]:
balanceData_a1b = balanceData.copy()
balanceData_a1b.reset_index(inplace=True)
balanceData_a1b = balanceData_a1b[balanceData_a1b['year'].isin([2015,2100])]
balanceData_a1b.drop(['dl', 'dc', 'sl', 'sc', 'pl', 'pc', 'pw', 'in', 'sigma', 'surpl_pr', 'surpl_co', 'surpl_tot', 'st_or',],axis=1, inplace=True) # we don't want these columns never
balanceData_a1b = balanceData_a1b[balanceData_a1b['scen'].isin(scenarios_a1b)]
balanceData_a1b = balanceData_a1b.groupby(['region','prod','prodLev','year']).agg(np.mean)
balanceData_a1b.reset_index(inplace=True)
balanceData_a1b = balanceData_a1b.groupby(['region','prodLev','year']).agg(np.sum)
balanceData_a1b = balanceData_a1b.unstack(['year']) # no inplace for unstack()
balanceData_a1b = pd.concat([balanceData_a1b,rtData_a1b_in,rtData_a1b_out], axis=1)
balanceData_a1b = balanceData_a1b.swaplevel('year','parName', axis=1)
balanceData_a1b = balanceData_a1b.unstack(['prodLev']) # temporary unstack to compute the France totals
balanceData_a1b.loc['France'] = balanceData_a1b.sum()
balanceData_a1b = balanceData_a1b.stack(['prodLev']) # temporary unstack to compute the France totals
balanceData_a1b[2015, 'avg_prod_price'] = balanceData_a1b[2015, 'rev'] / balanceData_a1b[2015, 'st']
balanceData_a1b[2100, 'avg_prod_price'] = balanceData_a1b[2100, 'rev'] / balanceData_a1b[2100, 'st']
balanceData_a1b[2015, 'avg_loc_price'] = balanceData_a1b[2015, 'exp'] / balanceData_a1b[2015, 'dt']
balanceData_a1b[2100, 'avg_loc_price'] = balanceData_a1b[2100, 'exp'] / balanceData_a1b[2100, 'dt']
balanceData_a1b.sortlevel('year', axis=1, inplace=True)
balanceData_a1b.drop([(2015,'rev'),(2100,'rev')],axis=1, inplace=True) 
balanceData_a1b.drop([(2015,'exp'),(2100,'exp')],axis=1, inplace=True) 

balanceData_a1b.columns.set_levels([ [2015, 2100], ['Imports','Consumption','Revenues','Expenses','Regional imports','Regional trade','Exports','Production','Producer price','Local price']], inplace=True)
balanceData_a1b[2015,'Total Supply'] = balanceData_a1b[2015,'Production'] + balanceData_a1b[2015,'Imports']
balanceData_a1b[2100,'Total Supply'] = balanceData_a1b[2100,'Production'] + balanceData_a1b[2100,'Imports']
balanceData_a1b[2015,'Total Demand'] = balanceData_a1b[2015,'Consumption'] + balanceData_a1b[2015,'Exports']
balanceData_a1b[2100,'Total Demand'] = balanceData_a1b[2100,'Consumption'] + balanceData_a1b[2100,'Exports']
balanceData_a1b.sortlevel('year', axis=1, inplace=True)
balanceData_a1b=balanceData_a1b.reindex_axis(['Production','Imports','Total Supply','Consumption','Exports','Total Demand','Regional trade', 'Producer price','Local price'], axis=1, level='parName')


print ("Notes: Balance doesn't close as byproduct production of industrial wood is not included in the supply.")
print ("Due to the increased worldwide demand and better availability of national resources, exports of primary products increase from 7.65% of the production in 2015 to 21.00% in 2100, while imports drop from 20.45% of consumption to 13.17%.")

balanceData_a1b_fr = balanceData_a1b.xs('France', level='region').copy()
#balanceData_a1b_fr = balanceData_a1b_fr.reset_index()
balanceData_a1b_fr = balanceData_a1b_fr.stack(['parName'])
balanceData_a1b_fr = balanceData_a1b_fr.unstack(['prodLev'])

#balanceData_a1b.columns.set_levels([ [2015, 2100], ['Primary Products','Secondary Products']], inplace=True)

balanceData_a1b_fr[2015,'All products'] = balanceData_a1b_fr[2015,'priProd']+balanceData_a1b_fr[2015,'secProd']
balanceData_a1b_fr[2100,'All products'] = balanceData_a1b_fr[2100,'priProd']+balanceData_a1b_fr[2100,'secProd']
#balanceData_a1b_fr = balanceData_a1b_fr.reindex_axis(['priProd','secProd','All products'], axis=1, level='prodLev')
balanceData_a1b_fr.sortlevel('year', axis=1, inplace=True)
balanceData_a1b_fr.loc[['Producer price'],(2015,'All products')] = np.nan
balanceData_a1b_fr.loc[['Producer price'],(2100,'All products')] = np.nan
balanceData_a1b_fr.loc[['Local price'],(2015,'All products')] = np.nan
balanceData_a1b_fr.loc[['Local price'],(2100,'All products')] = np.nan
balanceData_a1b_fr.columns.set_levels([ [2015, 2100], ['Prim. Products','Transf. Products','Total']], inplace=True)
#balanceData_a1b_fr.stack(['prodLevel'])
#balanceData_a1b_fr['All products'] = balanceData_a1b_fr.sum()
print(balanceData_a1b_fr.to_latex(formatters=[form_f2, form_f2, form_f2, form_f2, form_f2, form_f2]))
balanceData_a1b_fr



#pvt

Notes: Balance doesn't close as byproduct production of industrial wood is not included in the supply.
Due to the increased worldwide demand and better availability of national resources, exports of primary products increase from 7.65% of the production in 2015 to 21.00% in 2100, while imports drop from 20.45% of consumption to 13.17%.
\begin{tabular}{lrrrrrr}
\toprule
year &           2015 &                  &        &           2100 &                  &        \\
prodLev & Prim. Products & Transf. Products &  Total & Prim. Products & Transf. Products &  Total \\
parName        &                &                  &        &                &                  &        \\
\midrule
Production     &          48.20 &            39.41 &  87.61 &          63.96 &            44.18 & 108.14 \\
Imports        &           0.00 &            10.13 &  10.13 &           0.00 &             6.70 &   6.70 \\
Total Supply   &          48.20 &            49.54 &  97.74 &          63.96 &            50.88 

year                     2015                                        2100  \
prodLev        Prim. Products Transf. Products       Total Prim. Products   
parName                                                                     
Production          48.201538        39.406806   87.608344      63.959265   
Imports              0.000000        10.131906   10.131906       0.000000   
Total Supply        48.201538        49.538711   97.740249      63.959265   
Consumption         50.233006        49.538707   99.771714      57.205188   
Exports              3.685130         0.000000    3.685130      13.431031   
Total Demand        53.918136        49.538707  103.456843      70.636219   
Regional trade       3.350996        11.994425   15.345421       3.318933   
Producer price      36.607626       120.066585         NaN      52.862887   
Local price         37.753620       132.899951         NaN      29.779929   

year                                         
prodLev        Transf. Products       Total  
parName                                      
Production            44.176219  108.135485  
Imports                6.699716    6.699716  
Total Supply          50.875935  114.835200  
Consumption           50.875938  108.081126  
Exports                0.000000   13.431031  
Total Demand          50.875938  121.512157  
Regional trade        13.723087   17.042020  
Producer price       129.685088         NaN  
Local price          118.598400         NaN

#### Total surplus, producer surplus and consumer surplus: table by region now and 2100 [M€] ####

In [20]:
# Note: this results strongly depends on the treeshold in the integration of the demand function...
surplusData_a1b = balanceData.copy()
surplusData_a1b.reset_index(inplace=True)
surplusData_a1b = surplusData_a1b[surplusData_a1b['year'].isin([2015,2100])]
surplusData_a1b.drop(['dl','dc','da','dt','sl','sc','st','sa','pl','pc','pw','in','sigma', 'rev', 'st_or', 'exp'],axis=1, inplace=True) # we don't want these columns never
surplusData_a1b = surplusData_a1b[surplusData_a1b['scen'].isin(scenarios_a1b)]
surplusData_a1b = surplusData_a1b.groupby(['region','prod','prodLev','year']).agg(np.mean) # here it's an average
surplusData_a1b.reset_index(inplace=True)
surplusData_a1b = surplusData_a1b.fillna(0).groupby(['region','year']).agg(np.sum)            # here it's a sum
surplusData_a1b = surplusData_a1b.unstack(['year']) 
surplusData_a1b.loc['France'] = surplusData_a1b.sum()
surplusData_a1b.columns.set_levels([['producer surplus', 'consumer surplus', 'total surplus'], [2015, 2100]], inplace=True)
surplusData_a1b = surplusData_a1b/1000000
surplusData_a1b['producer surplus','diff_pc'] = 100*(surplusData_a1b['producer surplus',2100]-surplusData_a1b['producer surplus',2015])/surplusData_a1b['producer surplus',2015]
surplusData_a1b['consumer surplus','diff_pc'] = 100*(surplusData_a1b['consumer surplus',2100]-surplusData_a1b['consumer surplus',2015])/surplusData_a1b['consumer surplus',2015]
surplusData_a1b['total surplus','diff_pc'] = 100*(surplusData_a1b['total surplus',2100]-surplusData_a1b['total surplus',2015])/surplusData_a1b['total surplus',2015]
surplusData_a1b.columns.set_levels([['Producer surplus', 'Consumer surplus', 'Total surplus'], [2015, 2100, '% Diff']], inplace=True)
surplusData_a1b.sortlevel('parName', axis=1, inplace=True)
print("Notes: ")
print("- Large gains of both producer and consumer surplus, but the quantification of the latter depends on the choosen funcional form and the subjective limits placed to avoid inclusion of consumers with unrealistically high surplus.")
print("- However, while values are in real terms, future values are not discounted.")
print("- The gain of producer surplus is much more heterogeneous across regions than the gain of consumer surplus and more concentrated over the traditional forest producing regions.")
print(surplusData_a1b.to_latex(formatters=[form_f2, form_f2, form_f2, form_f2, form_f2, form_f2, form_f2, form_f2, form_f2]))
surplusData_a1b 


Notes: 
- Large gains of both producer and consumer surplus, but the quantification of the latter depends on the choosen funcional form and the subjective limits placed to avoid inclusion of consumers with unrealistically high surplus.
- However, while values are in real terms, future values are not discounted.
- The gain of producer surplus is much more heterogeneous across regions than the gain of consumer surplus and more concentrated over the traditional forest producing regions.
\begin{tabular}{lrrrrrrrrr}
\toprule
parName & Producer surplus &          &        & Consumer surplus &          &        & Total surplus &          &        \\
year &             2015 &     2100 & \% Diff &             2015 &     2100 & \% Diff &          2015 &     2100 & \% Diff \\
region &                  &          &        &                  &          &        &               &          &        \\
\midrule
AL     &            24.00 &    73.94 & 208.12 &           111.29 &   124.16 &  11.56 &     

parName Producer surplus                          Consumer surplus  \
year                2015         2100      % Diff             2015   
region                                                               
AL             23.996315    73.936252  208.115019       111.293868   
AQ             68.792386   212.969405  209.582816       178.351302   
AU             33.611284    70.661188  110.230552        73.246531   
BN             21.911490    26.414001   20.548630        82.196194   
BO             59.363860   120.949315  103.742337        95.046376   
BR             37.749392    51.591260   36.667791       193.862874   
CA             69.112448    99.231475   43.579743        84.409509   
CE             54.539129   110.661677  102.903272       153.492452   
CO              3.351876     4.960674   47.996935        16.082127   
FC             47.177587   117.133293  148.281654        64.053150   
HN             30.415674    44.818350   47.352812       116.631691   
IF             25.671758    30.651762   19.398768      1285.698739   
LI             39.433920    77.299125   96.021915        39.276846   
LO             67.410658   149.392489  121.615534       125.171602   
LR             23.473228    36.584753   55.857357       131.789926   
MP             36.018738    96.363645  167.537541       167.958424   
NP             13.997869    25.063820   79.054536       132.520725   
PA             56.124470    75.122021   33.848962       322.093026   
PC             18.250378    40.604146  122.483865        95.673654   
PI             57.507356    43.558363  -24.256015       105.038936   
PL             39.289716    59.188520   50.646342       218.715664   
RA             79.781308   142.448822   78.549119       424.737576   
France        906.980838  1709.604357   88.493989      4217.341192   

parName                         Total surplus                          
year            2100     % Diff          2015         2100     % Diff  
region                                                                 
AL        124.164763  11.564783    135.290184   198.101015  46.426747  
AQ        201.570910  13.019029    247.143688   414.540315  67.732512  
AU         81.735219  11.589203    106.857815   152.396407  42.616062  
BN         93.225080  13.417757    104.107684   119.639081  14.918589  
BO        106.375794  11.919885    154.410235   227.325109  47.221529  
BR        219.279737  13.110743    231.612265   270.870997  16.950195  
CA         95.123330  12.692671    153.521956   194.354805  26.597400  
CE        171.988404  12.050072    208.031581   282.650081  35.868833  
CO         17.761272  10.441064     19.434003    22.721947  16.918506  
FC         71.346358  11.386181    111.230737   188.479651  69.449251  
HN        132.626686  13.714106    147.047365   177.445036  20.672027  
IF       1441.753509  12.137740   1311.370497  1472.405272  12.279884  
LI         44.605279  13.566347     78.710766   121.904404  54.876405  
LO        139.918759  11.781552    192.582259   289.311248  50.227362  
LR        148.985711  13.047875    155.263154   185.570463  19.519962  
MP        186.704280  11.161009    203.977162   283.067925  38.774323  
NP        150.052504  13.229462    146.518595   175.116324  19.518157  
PA        362.540040  12.557556    378.217497   437.662061  15.717032  
PC        107.683667  12.553105    113.924031   148.287813  30.163769  
PI        119.676105  13.934994    162.546292   163.234469   0.423373  
PL        246.787471  12.834840    258.005380   305.975990  18.592872  
RA        474.654754  11.752475    504.518884   617.103576  22.315258  
France   4738.559632  12.358935   5124.322031  6448.163989  25.834480

### BY-SCENARIO analysis ###
#### producer/consumer/total surplus for constant, arpege_a1b_nopr, arpege_a1b_noph, arpege_a1b ####

In [21]:
surplusData = balanceData.copy()
surplusData.reset_index(inplace=True)
surplusData = surplusData[surplusData['year'].isin([2015,2100])]
surplusData.drop(['dl','dc','da','dt','sl','sc','st','sa','pl','pc','pw','in','sigma', 'rev', 'st_or', 'exp'],axis=1, inplace=True) # we don't want these columns never
surplusData = surplusData[surplusData['scen'].isin(['constant', 'arpege_a1b', 'arpege_a1b_nopr', 'arpege_a1b_noph'])]
surplusData = surplusData.fillna(0).groupby(['scen','year']).agg(np.sum) # this "clean" (make effective) the previous drop of columns
surplusData = surplusData.unstack(['year'])
surplusData = surplusData/1000000
surplusData.columns.set_levels([['Producers surplus', 'Consumers surplus', 'Total surplus'], [2015, 2100]], inplace=True)
print("Notes:")
print("- physical impact of climate change slightly positively impact producer surplus (faster growth?)")
print("- price cc impact (higher prices) more than double producer surplus")
print("- neither significantly impacts consumers surplus")
print(surplusData.to_latex(formatters=[form_i, form_i, form_i, form_i,form_i, form_i,]))

surplusData 

Notes:
- physical impact of climate change slightly positively impact producer surplus (faster growth?)
- price cc impact (higher prices) more than double producer surplus
- neither significantly impacts consumers surplus
\begin{tabular}{lrrrrrr}
\toprule
parName & Producers surplus &       & Consumers surplus &       & Total surplus &       \\
year &              2015 &  2100 &              2015 &  2100 &          2015 &  2100 \\
scen            &                   &       &                   &       &               &       \\
\midrule
arpege\_a1b      &               907 & 1,711 &             4,217 & 4,752 &         5,124 & 6,463 \\
arpege\_a1b\_noph &               906 & 1,739 &             4,217 & 4,719 &         5,123 & 6,458 \\
arpege\_a1b\_nopr &               905 &   787 &             4,222 & 4,124 &         5,127 & 4,911 \\
constant        &               904 &   765 &             4,221 & 4,095 &         5,125 & 4,860 \\
\bottomrule
\end{tabular}



parName         Producers surplus              Consumers surplus               \
year                         2015         2100              2015         2100   
scen                                                                            
arpege_a1b             907.020371  1710.661662       4217.431854  4752.143330   
arpege_a1b_noph        906.230756  1739.347315       4216.673472  4719.016300   
arpege_a1b_nopr        904.662623   787.307513       4222.134926  4124.085612   
constant               903.870338   764.943653       4221.384300  4094.767492   

parName         Total surplus               
year                     2015         2100  
scen                                        
arpege_a1b        5124.452225  6462.804992  
arpege_a1b_noph   5122.904228  6458.363616  
arpege_a1b_nopr   5126.797549  4911.393124  
constant          5125.254638  4859.711145

#### Role of expectations: distributive effects of for manager expectations over the production chain: cumulative 2015-2100 producer/consumer/total surplus and cumulative forest revenues (sum by pp of pw_pp*st_pp) ####

In [22]:
surplusData = balanceData.copy()
surplusData.reset_index(inplace=True)
surplusData = surplusData[surplusData['year'].isin([y for y in range(2015,2101)])]
#surplusData = surplusData[surplusData['year'].isin([2080])]
surplusData.drop(['dl','dc','da','dt','sl','sc','st','sa','pl','pc','pw','in','sigma', 'rev', 'st_or'],axis=1, inplace=True) # we don't want these columns never
surplusData = surplusData[surplusData['scen'].isin(['arpege_a1b', 'arpege_a1b_noexp', 'arpege_a1b_fullexp'])]

surplusData = surplusData.fillna(0).groupby(['scen']).agg(np.sum) 
surplusData.drop('year', axis=1, inplace=True)
#surplusData.columns.set_levels([['producers surplus', 'consumers surplus', 'total surplus'], [2015, 2100]], inplace=True)
print("Notes:")
print("- The value of the standing forest at 2100, that should also be added, is not included here.")
print("- Expectations of forest managers seems not to play any role in terms of hwp producers/consumers surpluses.")
print("- IMPORTANT: In previous reports, myopic/perfect foresight agens was in reference to the sole physical impact, and myopic agents were expecting higher returns than those that anticipated cc. Now it includes also prices, and are perfect foresights that have the highest expected returns.")
surplusData 

Notes:
- The value of the standing forest at 2100, that should also be added, is not included here.
- Expectations of forest managers seems not to play any role in terms of hwp producers/consumers surpluses.
- IMPORTANT: In previous reports, myopic/perfect foresight agens was in reference to the sole physical impact, and myopic agents were expecting higher returns than those that anticipated cc. Now it includes also prices, and are perfect foresights that have the highest expected returns.


parName,exp,surpl_pr,surpl_co,surpl_tot
scen,,,,
arpege_a1b,677267.026741,1.140901e+11,3.866176e+11,5.007076e+11
arpege_a1b_fullexp,677503.263292,1.140573e+11,3.866727e+11,5.007300e+11
arpege_a1b_noexp,677457.568377,1.140231e+11,3.866567e+11,5.006797e+11


In [23]:
balanceData_exp = balanceData.copy()
balanceData_exp.reset_index(inplace=True)
balanceData_exp = balanceData_exp[balanceData_exp['year'].isin([2100])]
balanceData_exp.drop(['dl', 'dc', 'sl', 'sc', 'sa', 'pl', 'pc', 'pw', 'in', 'rev', 'sigma', 'surpl_pr', 'surpl_co', 'surpl_tot', 'st_or','exp'],axis=1, inplace=True) # we don't want these columns never
balanceData_exp = balanceData_exp[balanceData_exp['scen'].isin(['arpege_a1b', 'arpege_a1b_noexp', 'arpege_a1b_fullexp'])]
balanceData_exp = balanceData_exp[balanceData_exp['prod'].isin(['hardWSawnW', 'softWSawnW'])]
balanceData_exp = balanceData_exp.groupby(['scen','prod']).agg(np.sum)
balanceData_exp.drop(['year'],axis=1, inplace=True) 
balanceData_exp = balanceData_exp.unstack(['prod'])
balanceData_exp = balanceData_exp.swaplevel('parName','prod', axis=1)
balanceData_exp.sortlevel('prod', axis=1, inplace=True)
balanceData_exp.columns.set_levels([['Hardwood sawnwood', 'Softwood sawnwood'], ['Imports','Consumption','Production']], inplace=True)
balanceData_exp

prod               Hardwood sawnwood                        Softwood sawnwood  \
parName                      Imports Consumption Production           Imports   
scen                                                                            
arpege_a1b                  0.388212    2.283800   1.895589          1.957443   
arpege_a1b_fullexp          0.388212    2.283769   1.895557          1.958456   
arpege_a1b_noexp            0.388212    2.283423   1.895212          1.957604   

prod                                       
parName            Consumption Production  
scen                                       
arpege_a1b           10.344446   8.387012  
arpege_a1b_fullexp   10.333312   8.374856  
arpege_a1b_noexp     10.342765   8.385149

In [24]:
# So no effects of expectations on either surplus nor sawnwood volumes.. let's see the area and volumes of broadleaved/coniferous in 2100..
forArea_exp = forArea_byGroup
forArea_exp = forArea_exp[forArea_exp['scen'].isin(['arpege_a1b', 'arpege_a1b_noexp', 'arpege_a1b_fullexp'])]
forArea_exp = forArea_exp[forArea_exp['year'].isin([2100])]
forArea_exp.drop(['year'],axis=1, inplace=True) 
forArea_exp = forArea_exp.groupby(['scen','spGroup']).agg(np.sum)
forArea_exp = forArea_exp.unstack(['spGroup'])
forArea_exp.columns = forArea_exp.columns.droplevel() # DROP TOP COLUMN LEVEL
#forArea_exp.columns.set_levels([ [2015, 2100], ['br', 'con']], inplace=True)

forArea_exp['Broadleaved'] = 100 * forArea_exp['broadleaved']  / (forArea_exp['broadleaved']+forArea_exp['coniferous']) 
forArea_exp['Coniferous'] = 100 * forArea_exp['coniferous']  / (forArea_exp['broadleaved']+forArea_exp['coniferous']) 
forArea_exp.drop(['broadleaved','coniferous'], axis=1, inplace=True)

forArea_exp

KeyError: 'broadleaved'

In [ ]:
# Let's see if at least in the regeneration (yearly avg) we see differences..

forReg = forestDataReg[forestDataReg['parName'] == 'regArea'].copy()
forReg = forReg[forReg['scen'].isin(['arpege_a1b', 'arpege_a1b_noexp', 'arpege_a1b_fullexp'])]
forReg = forReg[forReg['year'].isin([y for y in range(2015,2100)])]

forReg = forReg.drop(['freeDim','parName','forType'],axis=1) # we don't want these columns never

forReg = forReg.groupby(['scen','spGroup','year']).agg(np.sum)
forReg.reset_index(inplace=True)
forReg = forReg.groupby(['scen','spGroup']).agg(np.mean)

forReg.drop(['region','year'], axis=1, inplace=True)
forReg  = forReg.unstack(['spGroup'])
forReg.columns = forReg.columns.droplevel()
forReg = forReg/1000 # So to have 1000 Ha
print("Notes:")
print ("- even if relativelly large differneces in the expected returns, profitability difference between coniferous and broadleaved remains and due to the high transaction costs there is little impact.")
forReg

# Still low difference, cc not enought big to make many people chage their behaviours. Difference is larger
# if we consider only the first period, as after 2100 there is no longer any difference, the scenarios show a slighly
# larger heterogeneity


#### Forest management: influences on the surpluses downhill the production chain (table production/consumer/total surplus managed/unmanaged forest) , to the markets (hardwood vs softwood), to the forest (area br/con) and to the environment (carbon balance)  ####

In [ ]:
surplusData_fa = balanceData.copy()
surplusData_fa.reset_index(inplace=True)
surplusData_fa = surplusData_fa[surplusData_fa['year'].isin([y for y in range(2015,2101)])]
#surplusData = surplusData[surplusData['year'].isin([2080])]
surplusData_fa.drop(['dl','dc','da','dt','sl','sc','st','sa','pl','pc','pw','in','sigma', 'rev', 'st_or', 'exp'],axis=1, inplace=True) # we don't want these columns never
surplusData_fa = surplusData_fa[surplusData_fa['scen'].isin(['arpege_a1b','arpege_a1b_nomgm'])]
surplusData_fa = surplusData_fa.fillna(0).groupby(['scen']).agg(np.sum) 
surplusData_fa.drop(['year'], axis=1, inplace=True)
#surplusData.columns.set_levels([['producers surplus', 'consumers surplus', 'total surplus'], [2015, 2100]], inplace=True)
print("Notes:")
print("- management doesn't seems to effect hwp agents surpluses")
surplusData_fa

In [ ]:
# Let's see broadeaved/coniferous area % for unmaneged scenario:
forArea_fa = forArea_byGroup
forArea_fa = forArea_fa[forArea_fa['scen'].isin(['arpege_a1b', 'arpege_a1b_nomgm','constant'])]
forArea_fa = forArea_fa[forArea_fa['year'].isin([2100])]
forArea_fa.drop(['year'],axis=1, inplace=True) 
forArea_fa = forArea_fa.groupby(['scen','spGroup']).agg(np.sum)
forArea_fa = forArea_fa.unstack(['spGroup'])
forArea_fa.columns = forArea_fa.columns.droplevel() # DROP TOP COLUMN LEVEL


forArea_fa['Broadleaved'] = 100 * forArea_fa['broadleaved']  / (forArea_fa['broadleaved']+forArea_fa['coniferous']) 
forArea_fa['Coniferous'] = 100 * forArea_fa['coniferous']  / (forArea_fa['broadleaved']+forArea_fa['coniferous']) 
forArea_fa.drop(['broadleaved','coniferous'], axis=1, inplace=True)

print ("Note:")
print("Compared to constant clima/mgm, cc favour broadleaved. If then we remove mgm, even more.")
forArea_fa

In [ ]:
# Harwood/softwood HWP in function of intensity of management
balanceData_nomgm = balanceData.copy()
balanceData_nomgm.reset_index(inplace=True)
balanceData_nomgm = balanceData_nomgm[balanceData_nomgm['year'].isin([2100])]
balanceData_nomgm.drop(['dl', 'dc', 'sl', 'sc', 'sa', 'pl', 'pc', 'pw', 'in', 'rev', 'sigma', 'surpl_pr', 'surpl_co', 'surpl_tot', 'st_or', 'exp'],axis=1, inplace=True) # we don't want these columns never
balanceData_nomgm = balanceData_nomgm[balanceData_nomgm['scen'].isin(['arpege_a1b', 'arpege_a1b_nomgm'])]
balanceData_nomgm = balanceData_nomgm[balanceData_nomgm['prod'].isin(['hardWSawnW', 'softWSawnW'])]
balanceData_nomgm = balanceData_nomgm.groupby(['scen','prod']).agg(np.sum)
balanceData_nomgm.drop(['year'],axis=1, inplace=True) 
balanceData_nomgm = balanceData_nomgm.unstack(['prod'])
balanceData_nomgm = balanceData_nomgm.swaplevel('parName','prod', axis=1)
balanceData_nomgm.sortlevel('prod', axis=1, inplace=True)
balanceData_nomgm.columns.set_levels([['Hardwood sawnwood', 'Softwood sawnwood'], ['Imports','Consumption','Production']], inplace=True)

print("Notes:")
print("- without management slight decrease of softwood sawnwood production, partially compensated by increase in the imports.")
print(balanceData_nomgm.to_latex(formatters=[form_f3, form_f3, form_f3, form_f3, form_f3, form_f3]))

balanceData_nomgm

#### Graphical representation of the role of cc (first three diagrams b/c/tot) and mgm (last 3 diagrams) ####

In [ ]:
# Getting the values for printing the triangles
# forestDataReg:	scen 	parName 	region 	forType 	freeDim 	year 	value 	spGroup

area0 = forestDataReg[(forestDataReg['scen']=='arpege_a1b') & 
                      (forestDataReg['parName']=='forArea') &
                      (forestDataReg['year']==2015)
                     ]  # ha

#forArea_total_a1b = forArea_total[forArea_total['scen'].isin(scenarios_a1b)]

vol0  = forestDataReg[(forestDataReg['scen']=='arpege_a1b') & 
                      (forestDataReg['parName']=='vol') &
                      (forestDataReg['year']==2015)
                     ] # already in M m^3
area0_t =  area0.value.sum() / 1000000
vol0_t  =  vol0.value.sum()
area0_b =  area0[area0['spGroup']=='broadleaved'].value.sum() / 1000000
area0_c =  area0[area0['spGroup']=='coniferous'].value.sum()  / 1000000
vol0_b  =  vol0[vol0['spGroup']  =='broadleaved'].value.sum()
vol0_c  =  vol0[vol0['spGroup']  =='coniferous'].value.sum()

# scenario arpege_a1b, 2100
areat = forestDataReg[(forestDataReg['scen']=='arpege_a1b') & 
                      (forestDataReg['parName']=='forArea') &
                      (forestDataReg['year']==2100)
                     ]
volt  = forestDataReg[(forestDataReg['scen']=='arpege_a1b') & 
                      (forestDataReg['parName']=='vol') &
                      (forestDataReg['year']==2100)
                     ]
areat_t =  areat.value.sum() / 1000000
volt_t  =  volt.value.sum()
areat_b =  areat[areat['spGroup']=='broadleaved'].value.sum() / 1000000
areat_c =  areat[areat['spGroup']=='coniferous'].value.sum()  / 1000000
volt_b  =  volt[volt['spGroup']  =='broadleaved'].value.sum()
volt_c  =  volt[volt['spGroup']  =='coniferous'].value.sum()

# scenario constant, 2100
areac = forestDataReg[(forestDataReg['scen']=='constant') & 
                      (forestDataReg['parName']=='forArea') &
                      (forestDataReg['year']==2100)
                     ]
volc  = forestDataReg[(forestDataReg['scen']=='constant') & 
                      (forestDataReg['parName']=='vol') &
                      (forestDataReg['year']==2100)
                     ]
areac_t =  areac.value.sum() / 1000000
volc_t  =  volc.value.sum()
areac_b =  areac[areac['spGroup']=='broadleaved'].value.sum() / 1000000
areac_c =  areac[areac['spGroup']=='coniferous'].value.sum()  / 1000000
volc_b  =  volc[volc['spGroup']  =='broadleaved'].value.sum()
volc_c  =  volc[volc['spGroup']  =='coniferous'].value.sum()

# scenario arpege_a1b_nomgm, 2100
arean = forestDataReg[(forestDataReg['scen']=='arpege_a1b_nomgm') & 
                      (forestDataReg['parName']=='forArea') &
                      (forestDataReg['year']==2100)
                     ]
voln  = forestDataReg[(forestDataReg['scen']=='arpege_a1b_nomgm') & 
                      (forestDataReg['parName']=='vol') &
                      (forestDataReg['year']==2100)
                     ]
arean_t =  arean.value.sum() / 1000000
voln_t  =  voln.value.sum()
arean_b =  arean[arean['spGroup']=='broadleaved'].value.sum() / 1000000
arean_c =  arean[arean['spGroup']=='coniferous'].value.sum()  / 1000000
voln_b  =  voln[voln['spGroup']  =='broadleaved'].value.sum()
voln_c  =  voln[voln['spGroup']  =='coniferous'].value.sum()

"""
print("Starting points (broad/con/tot):")
print(area0_b)
print(area0_c)
print(area0_t)
print(vol0_b)
print(vol0_c)
print(vol0_t)

print("End of the total cc vector (broad/con/tot):")
print(areat_b)
print(areat_c)
print(areat_t)
print(volt_b)
print(volt_c)
print(volt_t)

print("End of the nocc vector (broad/con/tot):")
print(areac_b)
print(areac_c)
print(areac_t)
print(volc_b)
print(volc_c)
print(volc_t)

print("End of the nomgm vector (broad/con/tot):")
print(arean_b)
print(arean_c)
print(arean_t)
print(voln_b)
print(voln_c)
print(voln_t)
"""

print("Area/vol 2015->2100 (three diagrams for br/con/total) comparition between total cc effect (scenario arpege_a1b, black), constant dynamics (including managemnt, red) and, by diff, cc effect alone (green)")
# Broadleaved triangle..
# plotVectorChart_inner(origin,end1,endt,xlabel,ylabel,filename)
parserlib.plotVectorChart_inner((area0_b,vol0_b),(areac_b,volc_b),(areat_b,volt_b),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_cc_b.png", comp1_color='#8c0000', totcomp_color='#000080', diffcomp_color='#fe4040')
parserlib.plotVectorChart_inner((area0_c,vol0_c),(areac_c,volc_c),(areat_c,volt_c),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_cc_c.png", comp1_color='#8c0000', totcomp_color='#000080', diffcomp_color='#fe4040')
parserlib.plotVectorChart_inner((area0_t,vol0_t),(areac_t,volc_t),(areat_t,volt_t),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_cc_t.png", comp1_color='#8c0000', totcomp_color='#000080', diffcomp_color='#fe4040')



In [ ]:
print("Area/vol 2015->2100 (three diagrams for br/con/total) comparition between total cc effect (scenario arpege_a1b, black), cc effect without management (red) and, by diff, management effect alone (green)")
parserlib.plotVectorChart_inner((area0_b,vol0_b),(arean_b,volc_b),(areat_b,volt_b),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_mgm_b.png", comp1_color='#203b00', totcomp_color='#000080', diffcomp_color='#b6fe40')
parserlib.plotVectorChart_inner((area0_c,vol0_c),(arean_c,volc_c),(areat_c,volt_c),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_mgm_c.png", comp1_color='#203b00', totcomp_color='#000080', diffcomp_color='#b6fe40')
parserlib.plotVectorChart_inner((area0_t,vol0_t),(arean_t,volc_t),(areat_t,volt_t),"Area [M sq.km]", "Volume [M c.m.]","charts/vector_sums/vectors_mgm_t.png", comp1_color='#203b00', totcomp_color='#000080', diffcomp_color='#b6fe40')